In [7]:
#선생님의 W컨셉 크롤링 

# 선생님의 W컨셉 크롤링 

from selenium import webdriver  # 브라우저 컨트롤
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager  # 로컬 브라우저 버전과 매칭
import time, re
import pandas as pd

service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument("--headless")
# options.add_argument("--disable-gpu")  # 로컬의 그래픽카드를 쓰지 않는다 
options.add_argument("--disable-dev-shm-usage")  # 값이 많은 경우 로컬의 그래픽 쓰지마라
options.add_argument("--start-maximized")  # 데스크탑에 맞는 최적화 
options.add_argument("--window-size=1920x1080")  # <-- 오타: wubdiw → window
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")

driver = webdriver.Chrome(service=service, options=options)

# -- 여기까지가 기본 크롤링 옵션 

# → 베스트 클릭 → 의류, 가방, 신발, 엑세서리 까지 순회 → 버튼클릭 안쓰고 url 쓰고자함
# 딕셔너리를 쓰는 이유? 반복 순회할 때 키와 벨류값이 있다면 편하기 때문 

category_urls = {
    "여성의류": "https://display.wconcept.co.kr/rn/best?displayCategoryType=10101&gnbType=Y",
    "가방": "https://display.wconcept.co.kr/rn/best?displayCategoryType=10102&displaySubCategoryType=ALL&gnbType=Y",
    "신발": "https://display.wconcept.co.kr/rn/best?displayCategoryType=10103&displaySubCategoryType=ALL&gnbType=Y",
    "엑세서리": "https://display.wconcept.co.kr/rn/best?displayCategoryType=10104&displaySubCategoryType=ALL&gnbType=Y"
}

all_data = []  # 상단의 자료를 모두 받기 위한 변수 리스트 형식 <- 반복문 안에 갇히면 안됨 

# 즉 저기 이미지 url에서 숫자 파라미터만 뽑아와서 동일한 url 에 변하는 부분만 변수로 줘서 반복적으로 들어가면 좋음 → 함수로 만들면 좋겠지

def extract_product_id_from_imgsrc(src: str) -> str | None:
    if not src:
        return None
    m = re.search(r"/(\d+)(?:[_\.][^/?]+)?(?:\?.*)?$", src)
    return m.group(1) if m else None

def build_product_url(pid: str | None) -> str:
    return f"https://www.wconcept.co.kr/Product/{pid}" if pid else ""

# 이제 딕셔너리로 지정한 값을 반복해서 크롤러가 들어갈 수 있도록 반복문 설정 
for cat_name, url in category_urls.items():
    driver.get(url)
    time.sleep(3)

    for _ in range(10):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(2)

    products = driver.find_elements(By.CSS_SELECTOR, "div.product-item.ite.type-all")[0:100]

    for product in products:
        try:
            name = product.find_element(By.CSS_SELECTOR, "span.prdc-title span.text.detail").text.strip()

            try:
                img = product.find_element(By.CSS_SELECTOR, "button.area-img img")  # <-- 오타: prodcut → product
                src = img.get_attribute("src")
                pid = extract_product_id_from_imgsrc(src)
                url = build_product_url(pid)
            except:
                url = ""

            try:
                review_info = product.find_element(By.CSS_SELECTOR, "span.area-info span.text.cnt").text
                review_count = int(re.sub(r"[^\d]", "", review_info)) if review_info else 0  # <-- revieew_info 오타 수정 + 정규식 괄호 위치
            except:
                review_count = 0

            try:
                rating_text = product.find_element(By.CSS_SELECTOR, "span.area-info em.score").text
                review_avg = float(re.findall(r"[0-9.]+", rating_text)[0])
            except:
                review_avg = 0.0

            # all_data에 넣기
            all_data.append({
                "카테고리": cat_name,
                "상품명": name,
                "URL": url,
                "리뷰수": review_count,
                "평균별점": review_avg
            })

        except Exception as e:
            print("에러발생", e)
            continue

driver.quit()

print(all_data)


[]


In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time, re
import pandas as pd

service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--start-maximized")
options.add_argument("--window-size=1920x1080")
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")

driver = webdriver.Chrome(service=service, options=options)

category_urls = {
    "여성의류":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10101&displaySubCategoryType=ALL&gnbType=Y",
    "가방":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10102&displaySubCategoryType=ALL&gnbType=Y",
    "신발":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10103&displaySubCategoryType=ALL&gnbType=Y",
    "액세서리":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10104&displaySubCategoryType=ALL&gnbType=Y"
}

all_data = []

def extract_product_id_from_imgsrc(src: str) -> str | None :
    if not src:
        return None
    m = re.search(r"/(\d+)(?:[_\.][^/?]+)?(?:\?.*)?$", src)
    return m.group(1) if m else None

def build_product_url(pid: str | None) -> str :
    return f"https://www.wconcept.co.kr/Product/{pid}" if pid else ""

for cat_name, url in category_urls.items() :
    driver.get(url)
    time.sleep(3)

    for _ in range(10) :
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

    products = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")[:100]

    for product in products :
        try :
            name = product.find_element(By.CSS_SELECTOR, "span.prdc-title span.text.detail").text.strip()
            try :
                img = product.find_element(By.CSS_SELECTOR, "button.area-img img")
                src = img.get_attribute("src")
                pid = extract_product_id_from_imgsrc(src)
                url = build_product_url(pid)
            except :
                url = ""
            try :
                review_info = product.find_element(By.CSS_SELECTOR, "span.area-info span.text.cnt").text
                review_count = int(re.sub(r"[^\d]", "", review_info)) if review_info else 0
            except :
                review_count = 0
            try :
                rating_text = product.find_element(By.CSS_SELECTOR, "span.area-info em.score").text
                review_avg = float(re.findall(r"[0-9.]+", rating_text)[0])
            except :
                review_avg = 0.0

            all_data.append({
                "카테고리": cat_name,
                "상품명": name,
                "URL": url,
                "리뷰수": review_count,
                "평균평점": review_avg,
            })
                
        except Exception as e :
            print("에러발생 :", e)
            continue

driver.quit()

df = pd.DataFrame(all_data)

df_top100 = df.sort_values(by="리뷰수", ascending=False).head(100).reset_index()#정렬된 이후에 index값을 정렬하는데 정렬전의 값은 지워

category_summary = df.groupby("카테고리").agg({
    "리뷰수":"sum",
    "평균평점":"mean"
}).reset_index()

print(f"""
상위 100개 상품 :
{df_top100.head()}

카테고리별 통계 :
{category_summary}
""")



상위 100개 상품 :
   index  카테고리                                    상품명  \
0    200    신발                OORIGINAL BLACK - 조리 블랙   
1    123    가방  [14%쿠폰] [단독] Panier Petit Bag_6colors   
2    375  액세서리   [단독] Round Logo Ball Cap (10 Colors)   
3    302  액세서리           [단독]BT8032 티디 스퀘어 버클 벨트 18mm   
4     73  여성의류                       실켓 코튼 티셔츠_8color   

                                            URL   리뷰수  평균평점  
0  https://www.wconcept.co.kr/Product/301721294  8478   5.0  
1  https://www.wconcept.co.kr/Product/301756891  3730   5.0  
2  https://www.wconcept.co.kr/Product/301628486  3682   4.9  
3  https://www.wconcept.co.kr/Product/304460283  3131   4.9  
4  https://www.wconcept.co.kr/Product/301218440  2968   4.9  

카테고리별 통계 :
   카테고리    리뷰수   평균평점
0    가방  24218  4.267
1    신발  27342  4.281
2  액세서리  30519  4.016
3  여성의류  31521  4.435



In [ ]:
#엑셀로 가져오깅 

from openpyxl import Workbook

wb = Workbook()

ws_top100 = wb.active
ws_top100.title = "상위100상품"


ws_top100.append(["카테고리", "상품명", "url", "리뷰수", "평균평점"])

for idx, row in df_top100.iterrows() :
    ws_top100.append([row["카테고리"], row["상품명"], row["url"], row["리뷰수"], row["평균평점"]])

ws_category = wb.create_sheet(title="카테고리별 통계")